# ADC Mapping - 2D EPI

ADC mapping using a EPI 2D spin echo sequence for ADC mapping.

### Imports

In [ ]:
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import MRzeroCore as mr0
import numpy as np
import torch
from cmap import Colormap
from einops import rearrange
from mrpro.algorithms.reconstruction import DirectReconstruction
from mrpro.data import KData
from mrpro.data.traj_calculators import KTrajectoryIsmrmrd

from mrseq.scripts.adc_epi2d_se import main as create_seq
from mrseq.utils import combine_ismrmrd_files
from mrseq.utils import sys_defaults

### Settings
We are going to use a numerical phantom with a matrix size of 64 x 64 x 4.

In [ ]:
image_matrix_size = [64, 64]
repetition_time = 4
b_values = [0, 200, 400, 800]

tmp = tempfile.TemporaryDirectory()
fname_mrd = Path(tmp.name) / 'adc_epi2d_se.mrd'

### Create the digital phantom

We will use a 3D BrainWeb phantom which has to be downloaded first.

In [ ]:
!wget https://github.com/MRsources/MRzero-Core/raw/main/documentation/playground_mr0/subject05.npz

Now we can create the [MRzero](https://github.com/MRsources/MRzero-Core) phantom, but we set the B1-field to be constant everywhere.

In [ ]:
phantom = mr0.VoxelGridPhantom.brainweb('subject05.npz')
phantom = phantom.interpolate(image_matrix_size[0], image_matrix_size[1], 32)
phantom = phantom.slices([16])
phantom.B1[:] = 1.0

### Create the ADC EPI 2D spin echo sequence

To create the ADC EPI 2D spin echo sequence, we use the previously imported [adc epi2d script](../src/mrseq/scripts/adc_epi2d_se.py).


In [ ]:
sequence, fname_seq = create_seq(
    system=sys_defaults,
    test_report=False,
    timing_check=False,
    b_values=b_values,
    fov_xy=float(phantom.size.numpy()[0]),
    tr=repetition_time,
    n_slices=1,
    n_readout=image_matrix_size[0],
    n_phase_encoding=image_matrix_size[1],
    partial_fourier_factor=0.65,
)

### Simulate the sequence
Now, we pass the sequence and the phantom to the MRzero simulation and save the simulated signal as an (ISMR)MRD file.

In [ ]:
mr0_sequence = mr0.Sequence.import_file(str(fname_seq.with_suffix('.seq')))
signal, ktraj_adc = mr0.util.simulate(mr0_sequence, phantom, accuracy=1e0)
mr0.sig_to_mrd(fname_mrd, signal, sequence)
combine_ismrmrd_files(fname_mrd, Path(f'{fname_seq}_header.h5'))

### Reconstruct the images for different b-values
We use [MRpro](https://github.com/PTB-MR/MRpro) for the image reconstruction.

In [ ]:
kdata = KData.from_file(str(fname_mrd).replace('.mrd', '_with_traj.mrd'), trajectory=KTrajectoryIsmrmrd())
recon = DirectReconstruction(kdata, csm=None)
idata = recon(kdata)

We can now plot the images at different b-values.

In [ ]:
idat = idata.data.abs().numpy().squeeze()
fig, ax = plt.subplots(1, idat.shape[0], figsize=(4 * idat.shape[0], 4))
for bidx in range(idat.shape[0]):
    ax[bidx].imshow(idat[bidx, :, :], cmap='gray')
    ax[bidx].set_title(f'b-value = {b_values[bidx]} s/mm^2')
    ax[bidx].set_xticks([])
    ax[bidx].set_yticks([])

### Estimate the ADC maps
We use a linear fit of the logarithm of the signal to estimate the apparent diffusion coefficient. Afterward, we compare them to the input and ensure they match.

In [ ]:
log_signal = torch.log(idata.rss() / idata.rss()[0])

# Solve linear equation
b = torch.tensor(b_values, dtype=torch.float32)
A = torch.stack([b, torch.ones_like(b)], dim=1)
Y = log_signal.reshape(len(b), -1)
coef = torch.linalg.lstsq(A, Y).solution
m = coef[0].reshape(idata.shape[1:])
d_measured = -m.squeeze().numpy() * 1e3

d_input = np.roll(rearrange(phantom.D.numpy().squeeze()[::-1, ::-1], 'x y -> y x'), shift=(1, 1), axis=(0, 1))
obj_mask = np.zeros_like(d_input)
obj_mask[d_input > 0] = 1
d_measured = d_measured * obj_mask

fig, ax = plt.subplots(1, 3, figsize=(15, 3))
for cax in ax.flatten():
    cax.set_xticks([])
    cax.set_yticks([])

im = ax[0].imshow(d_input, vmin=0, vmax=1.5, cmap=Colormap('navia').to_mpl())
fig.colorbar(im, ax=ax[0], label='Input D (10^-3 * mm^2/s)')

im = ax[1].imshow(d_measured, vmin=0, vmax=1.5, cmap=Colormap('navia').to_mpl())
fig.colorbar(im, ax=ax[1], label='Measured D (10^-3 * mm^2/s)')

im = ax[2].imshow(d_measured - d_input, vmin=-1.5, vmax=1.5, cmap='bwr')
fig.colorbar(im, ax=ax[2], label='Difference D (10^-3 * mm^2/s)')

relative_error = np.sum(np.abs(d_input - d_measured)) / np.sum(np.abs(d_input))
print(f'Relative error {relative_error}')
assert relative_error < 0.2